#For the Naive model:

Close^t+1=Closet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
master_file = r"F:\SEM-7\Time Series Analysis\Project\Data\TSMC_Master_Features_15Y.csv"
tsm = pd.read_csv(master_file)

# Convert Date
tsm["Date"] = pd.to_datetime(tsm["Date"])
# Sort chronologically
tsm = tsm.sort_values("Date").reset_index(drop=True)

print("Master dataset loaded successfully")
print("Shape:", tsm.shape)
print("\nDate range:")
print(tsm["Date"].min(), "to", tsm["Date"].max())
print("\nColumns:")
print(tsm.columns.tolist())

Master dataset loaded successfully
Shape: (3771, 38)

Date range:
2011-09-12 00:00:00 to 2026-09-10 00:00:00

Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Log_Close', 'Return_1D', 'Log_Return_1D', 'High_Low_Range', 'Open_Close_Change', 'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3', 'Close_Lag_5', 'Close_Lag_10', 'Close_Lag_20', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'Return_Lag_5', 'Return_Lag_10', 'Return_Lag_20', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50', 'Volatility_5', 'Volatility_10', 'Volatility_20', 'Momentum_5D', 'Momentum_10D', 'Momentum_20D', 'Day_of_Week', 'Month', 'Quarter', 'Year']


In [ ]:
# CELL 2: NEXT-DAY TARGET

# Actual next trading day's closing price
tsm["Target_Close"] = tsm["Close"].shift(-1)
# Remove final row (no next-day value available)
tsm = tsm.dropna(subset=["Target_Close"]).reset_index(drop=True)
print("Target created successfully")
display(tsm[["Date", "Close", "Target_Close"]].head(10))

Target created successfully


,Date,Close,Target_Close
0,2011-09-12,11.94,11.96
1,2011-09-13,11.96,11.82
2,2011-09-14,11.82,12.11
3,2011-09-15,12.11,12.41
4,2011-09-16,12.41,12.22
5,2011-09-19,12.22,12.12
6,2011-09-20,12.12,11.92
7,2011-09-21,11.92,11.52
8,2011-09-22,11.52,11.71
9,2011-09-23,11.71,11.56


In [4]:
# TRAIN / VALIDATION / TEST SPLIT

n = len(tsm)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train = tsm.iloc[:train_end].copy()
validation = tsm.iloc[train_end:validation_end].copy()
test = tsm.iloc[validation_end:].copy()

print("DATASET SPLIT")
print(f"Total      : {len(tsm)} rows")
print(f"Train      : {len(train)} rows")
print(f"Validation : {len(validation)} rows")
print(f"Test       : {len(test)} rows")
print("\nDATE RANGES")

print(
    "Train      :",
    train["Date"].min().date(),
    "to",
    train["Date"].max().date()
)

print(
    "Validation :",
    validation["Date"].min().date(),
    "to",
    validation["Date"].max().date()
)

print(
    "Test       :",
    test["Date"].min().date(),
    "to",
    test["Date"].max().date()
)

DATASET SPLIT
Total      : 3770 rows
Train      : 2639 rows
Validation : 565 rows
Test       : 566 rows

DATE RANGES
Train      : 2011-09-12 to 2022-03-07
Validation : 2022-03-08 to 2024-06-05
Test       : 2024-06-06 to 2026-09-09


In [ ]:
# ============================================
# CELL 4: NAIVE FORECAST
# ============================================

# Naive prediction:
# Tomorrow's predicted Close = Today's actual Close
test["Naive_Prediction"] = test["Close"]
print("Naive forecasting completed.")
display(test[["Date", "Close", "Target_Close", "Naive_Prediction"]].head(10))

Naive forecasting completed.


,Date,Close,Target_Close,Naive_Prediction
3204,2024-06-06,162.070007,164.389999,162.070007
3205,2024-06-07,164.389999,168.160004,164.389999
3206,2024-06-10,168.160004,165.710007,168.160004
3207,2024-06-11,165.710007,172.979996,165.710007
3208,2024-06-12,172.979996,172.910004,172.979996
3209,2024-06-13,172.910004,172.509995,172.910004
3210,2024-06-14,172.509995,177.240005,172.509995
3211,2024-06-17,177.240005,179.690002,177.240005
3212,2024-06-18,179.690002,175.380005,179.690002
3213,2024-06-20,175.380005,173.960007,175.380005


In [ ]:
# CELL 5: NAIVE MODEL EVALUATION

# Actual and predicted values
y_true = test["Target_Close"]
y_pred = test["Naive_Prediction"]

# Metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

# Avoid division by zero for MAPE
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
r2 = r2_score(y_true, y_pred)

# Display results
print("NAIVE MODEL — TEST SET RESULTS")
print("=" * 60)
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.4f}%")
print(f"R²   : {r2:.4f}")

NAIVE MODEL — TEST SET RESULTS
MAE  : 5.2050
MSE  : 54.2798
RMSE : 7.3675
MAPE : 1.9653%
R²   : 0.9936


In [ ]:
# ============================================
# REUSABLE PROFITABILITY / BACKTEST FUNCTION
# ============================================

def backtest_strategy(data,prediction_column,initial_investment=100000,threshold=0.0):
    """
    Long-only Buy/Sell/Hold backtesting strategy.

    Parameters
    ----------
    data : DataFrame
        Must contain:
        Date, Close, prediction_column
    prediction_column : str
        Column containing next-day predicted Close.
    initial_investment : float
        Starting capital.
    threshold : float
        Minimum predicted return required for BUY/SELL.
        Example: 0.01 = 1%.

    Returns
    -------
    result : DataFrame
        Daily predictions, signals and portfolio values.
    summary : dict
        Profitability statistics.
    """

    result = data[["Date", "Close", prediction_column]].copy()
    result = result.reset_index(drop=True)

    # 1. Predicted return

    result["Predicted_Return"] = (result[prediction_column] - result["Close"])/result["Close"]

    # 2. Generate BUY / SELL / HOLD signals

    result["Signal"] = "HOLD"
    result.loc[result["Predicted_Return"] > threshold,"Signal"]  = "BUY"
    result.loc[result["Predicted_Return"] < -threshold,"Signal"] = "SELL"

    # 3. Portfolio simulation

    cash = initial_investment
    shares =0 .0
    portfolio_values = []
    transactions = []

    for i in range(len(result)):

        date   =  result.loc[i,"Date"]
        price  =  result.loc[i,"Close"]
        signal =  result.loc[i,"Signal"]

        # BUY
        if signal == "BUY" and shares == 0:

            shares = cash / price
            cash = 0

            transactions.append({
                "Date": date,
                "Action": "BUY",
                "Price": price,
                "Shares": shares
            })

        # SELL
        elif signal == "SELL" and shares > 0:

            cash = shares * price

            transactions.append({
                "Date": date,
                "Action": "SELL",
                "Price": price,
                "Shares": shares
            })

            shares = 0

        # Portfolio value
        portfolio_value = cash + (shares * price)

        portfolio_values.append(portfolio_value)


    # ----------------------------------------
    # 4. Store portfolio values
    # ----------------------------------------

    result["Portfolio_Value"] = portfolio_values


    # ----------------------------------------
    # 5. Close any open position at end
    # ----------------------------------------

    if shares > 0:

        final_price = result["Close"].iloc[-1]

        cash = shares * final_price

        transactions.append({
            "Date": result["Date"].iloc[-1],
            "Action": "FINAL_SELL",
            "Price": final_price,
            "Shares": shares
        })

        shares = 0


    # ----------------------------------------
    # 6. Final portfolio value
    # ----------------------------------------

    final_value = cash

    total_profit = final_value - initial_investment

    percentage_return = (
        total_profit / initial_investment
    ) * 100


    # ----------------------------------------
    # 7. Number of trades
    # ----------------------------------------

    transaction_df = pd.DataFrame(transactions)

    number_of_trades = len(transaction_df)


    # ----------------------------------------
    # 8. Buy & Hold benchmark
    # ----------------------------------------

    first_price = result["Close"].iloc[0]
    last_price = result["Close"].iloc[-1]

    buy_hold_shares = initial_investment / first_price

    buy_hold_final_value = buy_hold_shares * last_price

    buy_hold_profit = (
        buy_hold_final_value - initial_investment
    )

    buy_hold_return = (
        buy_hold_profit / initial_investment
    ) * 100


    # ----------------------------------------
    # 9. Summary
    # ----------------------------------------

    summary = {
        "Initial Investment": initial_investment,
        "Final Portfolio Value": final_value,
        "Total Profit/Loss": total_profit,
        "Strategy Return (%)": percentage_return,
        "Number of Transactions": number_of_trades,
        "Buy & Hold Final Value": buy_hold_final_value,
        "Buy & Hold Profit/Loss": buy_hold_profit,
        "Buy & Hold Return (%)": buy_hold_return
    }


    return result, transaction_df, summary

In [10]:
# ============================================
# NAIVE MODEL PROFITABILITY
# ============================================

naive_result, naive_transactions, naive_summary = backtest_strategy(
    data=test,
    prediction_column="Naive_Prediction",
    initial_investment=1000000,
    threshold=0.01
)

print("NAIVE MODEL — PROFITABILITY RESULTS")
print("=" * 60)

for key, value in naive_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:,.2f}")
    else:
        print(f"{key}: {value}")

NAIVE MODEL — PROFITABILITY RESULTS
Initial Investment: 1000000
Final Portfolio Value: 1000000
Total Profit/Loss: 0
Strategy Return (%): 0.00
Number of Transactions: 0
Buy & Hold Final Value: 2,686,246.47
Buy & Hold Profit/Loss: 1,686,246.47
Buy & Hold Return (%): 168.62
